# Шаг 1. Загрузка и очистка данных

**Проект:** отмены бронирований в отелях — прогноз и политика предоплаты.

**Цель шага:** получить очищенный датасет, в котором каждое решение по очистке посчитано и обосновано.

**Содержание:** 1. Загрузка · 2. Первичная проверка · 3. Проблемы и решения · 4. Дубли · 5. Запуск очистки и журнал · 6. Устойчивость к обработке дублей · 7. Признаки · 8. Гарантии · 9. Выводы

In [1]:
import pandas as pd

from src.data import load_raw
from src.features import LEAKAGE_COLUMNS
from src.pipeline import build_dataset, check_guarantees, compare_variants, save_outputs

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', None)

## 1. Загрузка

Датасет Hotel Booking Demand (Antonio, Almeida, Nunes, *Data in Brief*, 2019): брони двух отелей в Португалии — городского (Лиссабон) и курортного (Алгарви), заезды 07.2015–08.2017. По документации датасета одна строка — одна бронь; в §4 показываем, что одинаковые строки, скорее всего, — номера одной групповой брони.

In [2]:
raw = load_raw()
raw.shape

(119390, 32)

## 2. Первичная проверка

Типы, пропуски и число уникальных значений по каждой колонке.

In [3]:
DESCRIPTIONS = {
    'hotel': 'тип отеля: City Hotel или Resort Hotel',
    'is_canceled': 'бронь отменена (1) или нет (0) — целевая переменная',
    'lead_time': 'дней между датой бронирования и датой заезда',
    'arrival_date_year': 'год заезда',
    'arrival_date_month': 'месяц заезда (название)',
    'arrival_date_week_number': 'номер недели заезда (расходится с датой, см. §3.6)',
    'arrival_date_day_of_month': 'день месяца заезда',
    'stays_in_weekend_nights': 'число ночей в выходные дни',
    'stays_in_week_nights': 'число ночей в будние дни',
    'adults': 'число взрослых гостей',
    'children': 'число детей',
    'babies': 'число младенцев',
    'meal': 'тип забронированного пакета питания',
    'country': 'страна происхождения гостя (код)',
    'market_segment': 'сегмент продаж — группа клиентов, канал привлечения',
    'distribution_channel': 'канал продаж брони',
    'is_repeated_guest': 'гость уже бронировал раньше (1) или нет (0)',
    'previous_cancellations': 'число прошлых отмен этого гостя',
    'previous_bookings_not_canceled': 'число прошлых неотменённых броней гостя',
    'reserved_room_type': 'забронированный тип номера',
    'assigned_room_type': 'фактически присвоенный тип номера — известен при заезде',
    'booking_changes': 'число изменений брони после создания — известно постфактум',
    'deposit_type': 'тип депозита: без возврата, с возвратом, нет депозита',
    'agent': 'ID турагента (анонимный код)',
    'company': 'ID компании-заказчика (анонимный код)',
    'days_in_waiting_list': 'дней в листе ожидания до подтверждения',
    'customer_type': 'тип клиента: транзиент, группа, контракт и т. п.',
    'adr': 'средняя цена за номер за ночь, €',
    'required_car_parking_spaces': 'число нужных парковочных мест',
    'total_of_special_requests': 'число специальных пожеланий',
    'reservation_status': 'финальный статус брони — известен только по итогу',
    'reservation_status_date': 'дата финального статуса — известна только по итогу',
}


def _role(column):
    """Роль колонки: цель, утечка (LEAKAGE_COLUMNS), удаляется на шаге 11 или признак."""
    if column == 'is_canceled':
        return 'цель'
    if column in LEAKAGE_COLUMNS:
        return 'утечка'
    if column == 'arrival_date_week_number':
        return 'удаляется'
    return 'признак'


overview = pd.DataFrame({
    'тип': raw.dtypes.astype(str),
    'пропусков': raw.isna().sum(),
    'уникальных': raw.nunique(),
    'роль': [_role(col) for col in raw.columns],
    'описание': [DESCRIPTIONS[col] for col in raw.columns],
})
overview

,тип,пропусков,уникальных,роль,описание
hotel,str,0,2,признак,тип отеля: City Hotel или Resort Hotel
is_canceled,int64,0,2,цель,бронь отменена (1) или нет (0) — целевая переменная
lead_time,int64,0,479,признак,дней между датой бронирования и датой заезда
arrival_date_year,int64,0,3,признак,год заезда
arrival_date_month,str,0,12,признак,месяц заезда (название)
arrival_date_week_number,int64,0,53,удаляется,"номер недели заезда (расходится с датой, см. §3.6)"
arrival_date_day_of_month,int64,0,31,признак,день месяца заезда
stays_in_weekend_nights,int64,0,17,признак,число ночей в выходные дни
stays_in_week_nights,int64,0,35,признак,число ночей в будние дни
adults,int64,0,14,признак,число взрослых гостей


**Роли переменных.** Цель — `is_canceled`. Признаки брони: отель, срок брони (`lead_time`), дата заезда, ночи, гости, питание, страна, сегмент и канал продаж, история гостя, тип номера, депозит, агент, компания, лист ожидания, тип клиента, цена за ночь (`adr`), парковка, спецзапросы. **Утечки** — колонки, значения которых известны только после брони или в день заезда; в модель они не попадут:

In [4]:
list(LEAKAGE_COLUMNS)

['reservation_status',
 'reservation_status_date',
 'assigned_room_type',
 'booking_changes']

In [5]:
int(raw.duplicated().sum())

31994

**Вывод.** Пропуски — в `children`, `country`, `agent`, `company`; есть полностью одинаковые строки. Ниже — каждая проблема отдельно: что нашли, решение, почему.

## 3. Проблемы и решения

### 3.1. Ошибки в цене за ночь

In [6]:
raw['adr'].describe()

count    119390.000000
mean        101.831122
std          50.535790
min          -6.380000
25%          69.290000
50%          94.575000
75%         126.000000
max        5400.000000
Name: adr, dtype: float64

In [7]:
extremes = pd.concat([raw.nsmallest(2, 'adr'), raw.nlargest(3, 'adr')])
extremes[['hotel', 'adr', 'adults', 'stays_in_week_nights', 'market_segment']]

,hotel,adr,adults,stays_in_week_nights,market_segment
14969,Resort Hotel,-6.38,2,6,Groups
0,Resort Hotel,0.00,2,0,Direct
48515,City Hotel,5400.00,2,1,Offline TA/TO
111403,City Hotel,510.00,1,1,Offline TA/TO
15083,Resort Hotel,508.00,2,1,Corporate


In [8]:
int((raw['adr'] == 0).sum())

1959

**Решение:** удалить бронь с отрицательной ценой (шаг 1) и бронь за 5 400 € (шаг 2). **Почему:** цена не бывает отрицательной; 5 400 € в 10 раз дороже следующей брони (510 €) при 2 взрослых на 1 ночь — ошибки ввода. Цена 0 (1 959 броней по исходным данным) — реальные бесплатные и служебные брони: оставляем, помечаем флагом `is_free` и не используем в анализе цен; после удалений 1–6 и склейки дублей флаг `is_free` остаётся у 1 040 броней (см. §7).

### 3.2. Нереальное число гостей

In [9]:
raw.loc[(raw['adults'] > 10) | (raw['children'] >= 9) | (raw['babies'] >= 9),
        ['adults', 'children', 'babies', 'market_segment', 'is_canceled']]

,adults,children,babies,market_segment,is_canceled
328,2,10.0,0,Offline TA/TO,1
1539,40,0.0,0,Direct,1
1587,26,0.0,0,Offline TA/TO,1
1643,50,0.0,0,Direct,1
1752,26,0.0,0,Offline TA/TO,1
1884,26,0.0,0,Offline TA/TO,1
1917,27,0.0,0,Direct,1
1962,27,0.0,0,Direct,1
2003,26,0.0,0,Offline TA/TO,1
2164,26,0.0,0,Offline TA/TO,1


**Решение:** удалить (шаг 3). **Почему:** в один номер не поселить 10 детей или 55 взрослых — это ошибки ввода или служебные записи.

### 3.3. Брони на 0 ночей и без гостей

In [10]:
nights = raw['stays_in_weekend_nights'] + raw['stays_in_week_nights']
guests = raw['adults'] + raw['children'].fillna(0) + raw['babies']
pd.Series({
    'броней на 0 ночей': int((nights == 0).sum()),
    'из них с ценой 0, %': f"{100 * (raw.loc[nights == 0, 'adr'] == 0).mean():.1f}",
    'броней без гостей': int((guests == 0).sum()),
})

броней на 0 ночей        715
из них с ценой 0, %    100.0
броней без гостей        180
dtype: object

**Решение:** удалить (шаги 4 и 5). **Почему:** бронь без ночей — посещение на день, а не проживание; бронь без гостей не описывает реальное проживание. Объект исследования — номер, в котором собирались ночевать.

### 3.4. Категории Undefined

In [11]:
raw[['meal', 'market_segment', 'distribution_channel']].apply(lambda col: int(col.eq('Undefined').sum()))

meal                    1169
market_segment             2
distribution_channel       5
dtype: int64

**Решение:** питание `Undefined` объединить с `SC` (шаг 7); брони с сегментом или каналом `Undefined` удалить (шаг 6). **Почему:** по документации датасета `Undefined` и `SC` в питании — одно и то же («без пакета питания»); в сегменте и канале это единицы броней без смысла.

### 3.5. Пропуски: страна, агент, компания, дети

In [12]:
raw[['country', 'agent', 'company', 'children']].isna().sum()

country        488
agent        16340
company     112593
children         4
dtype: int64

In [13]:
agent_counts = raw['agent'].value_counts()
pd.Series({
    'агентов всего (по исходным данным)': int(raw['agent'].nunique()),
    'из них с < 100 бронями': int((agent_counts < 100).sum()),
    'их броней (по исходным данным)': int(agent_counts[agent_counts < 100].sum()),
})

агентов всего (по исходным данным)     333
из них с < 100 бронями                 249
их броней (по исходным данным)        5230
dtype: int64

**Решения:** пустая страна → категория `unknown` (шаг 8): пропуск информативен, страну часто уточняют при заезде. Пустые агент и компания — не пропуск, а «без агента / не от компании»: оставляем, добавляем флаги `has_agent`, `has_company`; агентов с < 100 бронями объединяем в `other` (шаг 9) — по исходным данным это 249 из 333 агентов и 5 230 броней; порог считается после склейки дублей (шаг 12, см. §5), поэтому база другая — 86 617 броней-решений, из них 4 534 помечены `other`. 4 пропуска в `children` → 0 (шаг 10): 0 — самое частое значение.

### 3.6. Номер недели заезда

In [14]:
months = {m: i for i, m in enumerate(pd.date_range('2016-01-01', periods=12, freq='MS').month_name(), 1)}
arrival = pd.to_datetime(pd.DataFrame({
    'year': raw['arrival_date_year'],
    'month': raw['arrival_date_month'].map(months),
    'day': raw['arrival_date_day_of_month'],
}))
mismatch = arrival.dt.isocalendar().week.to_numpy() != raw['arrival_date_week_number'].to_numpy()
round(100 * mismatch.mean(), 1)

np.float64(53.7)

**Решение:** удалить колонку (шаг 11), неделю при необходимости считать из даты. **Почему:** номер недели расходится с календарной неделей даты заезда — колонка противоречит дате.

## 4. Дубли — главное решение очистки

In [15]:
dup = raw.duplicated()
pd.Series({
    'лишних копий': int(dup.sum()),
    'доля отмен, все строки, %': f"{100 * raw['is_canceled'].mean():.1f}",
    'доля отмен без дублей, %': f"{100 * raw.drop_duplicates()['is_canceled'].mean():.1f}",
})

лишних копий                 31994
доля отмен, все строки, %     37.0
доля отмен без дублей, %      27.5
dtype: object

In [16]:
pd.DataFrame({
    'доля копий в сегменте, %': (100 * dup.groupby(raw['market_segment']).mean()).round(1),
    'строк': raw['market_segment'].value_counts(),
}).sort_values('доля копий в сегменте, %', ascending=False)

,"доля копий в сегменте, %",строк
market_segment,,
Groups,75.1,19811
Offline TA/TO,42.7,24219
Corporate,20.5,5295
Online TA,8.6,56477
Direct,6.4,12606
Complementary,5.5,743
Aviation,4.2,237
Undefined,0.0,2


In [17]:
(100 * dup.groupby(raw['deposit_type']).mean()).round(1)

deposit_type
No Deposit    17.6
Non Refund    92.9
Refundable    34.0
dtype: float64

In [18]:
group_sizes = raw.groupby(list(raw.columns), dropna=False).size()
group_sizes[group_sizes > 1].value_counts().sort_index().head(10)

2     5034
3      969
4      497
5      275
6      171
7      115
8      103
9       93
10     123
11      60
Name: count, dtype: int64

In [19]:
pairs = group_sizes[group_sizes == 2].reset_index()
online_direct_pairs = pairs[pairs['market_segment'].isin(['Online TA', 'Direct'])]
pd.Series({
    'пар-дублей в Online TA и Direct': len(online_direct_pairs),
    'из них с общим агентом, %': f"{100 * online_direct_pairs['agent'].notna().mean():.1f}",
})

пар-дублей в Online TA и Direct    3160
из них с общим агентом, %          92.3
dtype: object

**Что нашли.** Копии сосредоточены в групповых продажах (сегмент «Группы», офлайн-агентства) и в депозите «без возврата»; встречаются группы по 3–10+ одинаковых строк. ID брони авторы удалили при обезличивании, поэтому одинаковые строки — скорее **номера одной групповой брони**: одни даты, тариф и условия, а отмена — одно решение группы.

**Варианты:** A — удалить дубли (занижает отмены и стоимость бронирований, теряем реальные номера); B — оставить (одна групповая отмена считается десятком независимых — тесты завышают значимость); **C — склеить одинаковые строки в одну бронь-решение с числом номеров `n_rooms`**.

**Решение: C.** Тесты и модель работают с независимыми решениями, деньги считаются с весом `n_rooms`, ни один номер не теряется. **Риск:** ошибочно склеить две независимые брони, совпавшие по всем колонкам (вплоть до цены и срока брони до дня) — подозрительны пары в индивидуальных продажах (Online TA, Direct): таких пар 3 160, но у 92,3% из них общий агент, то есть это, скорее всего, один агент бронирует несколько номеров, а не совпадение двух разных гостей. **Страховка:** ниже сравниваем варианты A, B, C; в следующих шагах проверяем на них гипотезы и модель.

## 5. Запуск очистки и журнал

Каждый шаг записывает, сколько строк затронул и почему. Журнал сохраняется в `reports/cleaning_log.csv`.

In [20]:
clean_df, log = build_dataset(raw)
log.to_frame()

,step,action,rule,rows_before,rows_after,rows_affected,reason
0,10,modify,"children: пропуск → 0, тип int",119390,119390,4,Пропусков 4 из 119 тыс.; 0 — самое частое значение (93 %); дробный тип был только из-за пропусков.
1,1,remove,Отрицательная цена за ночь,119390,119389,1,"Цена не может быть отрицательной — ошибка ввода (в данных 1 такая бронь, −6,38 €)."
2,2,remove,Цена за ночь ≥ 5000 €,119389,119388,1,"Единственная бронь за 5 400 € в 10 раз дороже следующей (510 €), при этом 2 взрослых на 1 ночь — ошибка ввода."
3,3,remove,"Детей или младенцев > 8, взрослых > 10",119388,119373,15,"Столько гостей в один номер не поселить: 10 детей, 9–10 младенцев, 20–55 взрослых — ошибки ввода или служебные записи (15 броней)."
4,4,remove,Бронь на 0 ночей,119373,118658,715,"715 броней без единой ночи, все с ценой 0 — это посещения на день, а не проживание в номере."
5,5,remove,Бронь без гостей,118658,118548,110,"180 (по исходным данным) броней без взрослых, детей и младенцев не описывают реальное проживание."
6,6,remove,Сегмент или канал продаж Undefined,118548,118543,5,"Категория без смысла и слишком мала для отдельной группы: 2 брони по сегменту, 5 по каналу."
7,7,modify,Питание Undefined → SC,118543,118543,1160,"По документации датасета (Antonio et al., 2019) Undefined и SC означают одно: без пакета питания."
8,8,modify,Пустая страна → unknown,118543,118543,473,Пропуск информативен — страну часто уточняют при заезде; удаление стоило бы 488 (по исходным данным) броней.
9,11,modify,Удалена колонка arrival_date_week_number,118543,118543,118543,Номер недели не совпадает с календарной неделей даты заезда в 54 % строк; неделю считаем из даты.


**Как читать `rows_affected`.** Число в `reason` иногда взято из сырых данных, а `rows_affected` — после более ранних шагов. Например, в сырых данных 180 броней без гостей, но на шаге 5 `rows_affected` = 110: 70 из них уже удалены на шаге 4 как брони на 0 ночей.

**Почему шаги не по порядку номеров.** Шаг 10 (пропуски в `children`) выполняется первым, чтобы число гостей в шагах 3 и 5 считалось корректно. Шаг 9 (редкие агенты → `other`) выполняется после склейки дублей (шаг 12), чтобы не склеить разные брони одного агента в одну по ошибке.

In [21]:
pd.Series({
    'строк в исходных данных': len(raw),
    'удалено строк': log.rows_removed(),
    'номеров после очистки': int(clean_df['n_rooms'].sum()),
    'броней-решений после склейки': len(clean_df),
})

строк в исходных данных         119390
удалено строк                      847
номеров после очистки           118543
броней-решений после склейки     86617
dtype: int64

## 6. Устойчивость к обработке дублей

Доля отмен по броням и по номерам и доля отменённой стоимости бронирований для трёх вариантов. Варианты A/B/C считаются после удаления 847 ошибочных строк (раздел 3), поэтому числа здесь немного отличаются от раздела 4, где дубли считаны по сырым данным: там 31 994 дублирующихся строки и отмены 37,0% → 27,5%, а здесь после очистки — 31 926 склеенных строк. Порог редких агентов (< 100 броней, шаг 9) в каждом варианте считается на его собственных строках — на сравнение здесь это не влияет, `compare_variants` не использует `agent`.

In [22]:
compare_variants(raw).round(3)

,bookings,rooms,cancel_rate_bookings,cancel_rate_rooms,canceled_value_share
variant,,,,,
A,86617,86617,0.277,0.277,0.333
B,118543,118543,0.373,0.373,0.392
C,86617,118543,0.277,0.373,0.392


**Как читать.** По номерам вариант C совпадает с B — склейка ничего не теряет. Вариант A занижает долю отмен и отменённой стоимости бронирований. По броням-решениям доля отмен в C ниже, чем в B: групповые брони больше не умножаются на число номеров.

## 7. Признаки

| Признак | Зачем |
|---|---|
| `arrival_date`, `booking_date` | дата заезда и дата бронирования (заезд − `lead_time`) |
| `nights_week`, `nights_weekend`, `nights_total`, `weekend_share` | длительность и тип поездки |
| `guests`, `is_family` | размер компании, семья с детьми |
| `has_agent`, `has_company`, `agent_id` | бронь через агента / от компании; исходный ID агента |
| `is_free` | бесплатная бронь (цена 0) |
| `n_rooms`, `booking_value` | число номеров и стоимость брони = цена × ночи × номера |
| `arrival_month`, `arrival_weekday`, `season` | сезонность |

In [23]:
clean_df[['booking_key', 'hotel', 'arrival_date', 'booking_date', 'nights_total', 'weekend_share',
          'guests', 'is_family', 'n_rooms', 'booking_value', 'season', 'is_canceled']].head()

,booking_key,hotel,arrival_date,booking_date,nights_total,weekend_share,guests,is_family,n_rooms,booking_value,season,is_canceled
0,0,Resort Hotel,2015-07-01,2015-06-24,1,0.0,1,0,1,75.0,summer,0
1,1,Resort Hotel,2015-07-01,2015-06-18,1,0.0,1,0,1,75.0,summer,0
2,2,Resort Hotel,2015-07-01,2015-06-17,2,0.0,2,0,2,392.0,summer,0
3,3,Resort Hotel,2015-07-01,2015-07-01,2,0.0,2,0,1,214.0,summer,0
4,4,Resort Hotel,2015-07-01,2015-06-22,2,0.0,2,0,1,206.0,summer,0


In [24]:
int(clean_df['is_free'].sum())

1040

## 8. Гарантии очищенного датасета

In [25]:
guarantees = pd.Series(check_guarantees(clean_df, len(raw), log), name='выполнено')
assert guarantees.all(), guarantees[~guarantees]
guarantees

1. нет одинаковых строк, n_rooms ≥ 1              True
2. сумма n_rooms = строк после удалений           True
3. ночей ≥ 1, гостей ≥ 1, 0 ≤ adr < порога        True
4. нет Undefined в категориях                     True
5. пропуски только в agent, agent_id и company    True
6. журнал непрерывен                              True
Name: выполнено, dtype: bool

In [26]:
save_outputs(clean_df, log)

## 9. Выводы шага

- Из 119 390 строк исходных данных удалено 847 (0,7%): отрицательная цена за ночь (1), цена 5 400 € (1), нереальное число гостей (15), брони на 0 ночей (715), брони без гостей (110), сегмент или канал продаж Undefined (5).
- После удалений и склейки одинаковых строк осталось 118 543 номера и 86 617 броней-решений: 31 926 одинаковых строк склеены в `n_rooms` (считаем их номерами одной групповой брони, а не независимыми совпадениями).
- В варианте C доля отменённых номеров — 37,3% (как в B), доля отменённых броней-решений — 27,7% (как в A). Если по ошибке принять вариант A (с удалёнными дублями) за оценку доли отменённых номеров, она занижается почти на 10 п.п.: 27,7% вместо 37,3%. Так же вариант A занижает долю отменённой стоимости бронирований: 33,3% вместо 39,2% (как в B и C).
- Все 6 гарантий очищенного датасета выполнены: нет одинаковых строк и `n_rooms` ≥ 1, сумма `n_rooms` равна числу строк после удалений, ночи и гости ≥ 1, цена в допустимых пределах, нет категорий Undefined, пропуски остались только в `agent`, `agent_id` и `company`, журнал очистки непрерывен.
- В следующий шаг (EDA и проверка гипотез) передаются: очищенный датасет `data/processed/bookings_clean.parquet`, журнал очистки `reports/cleaning_log.csv` и функция `compare_variants` для сравнения вариантов обработки дублей A, B и C.